# PRIMO SCRIPT — Esplorazione di ADNIMERGE (Kaggle)

Obiettivo: applicare le FASI 1 e 2 del tuo documento di metodo
("Studio base, no modifiche" + "Prime modifiche base, conservativo")
su un dataset reale ADNI-like, senza toccare il DataLake.

NON devi scrivere la pipeline generica: qui si ESPLORA e si annota.
Quello che scopri (colonne, tipi, nulli, colonne chiave) servira' poi a
compilare lo schema di configurazione del file. Questo e' il primo passo.

COME SI USA:
- Questo e' un normale file .py, ma i marcatori "# %%" lo dividono in celle:
    in VSCode puoi eseguire una cella alla volta con Shift+Invio e vedere
    subito il risultato (come in un notebook). Vai cella per cella, dall'alto.
- Dove trovi "TODO" devi completare tu. Dove trovi codice gia' scritto,
    e' un esempio da cui copiare il modo di fare.

CRITERI DI "FATTO":
1) il file gira dall'inizio alla fine senza errori;
2) produce: shape, tipi, nulli per colonna, duplicati, distribuzione delle
     visite per soggetto, e un primo sguardo alle variabili categoriche;
3) nell'ultima cella scrivi COSA hai visto e COSA decideresti di fare
     (e perche'). Questa e' la parte piu' importante.

# 1. SETUP

Creiamo un ambiente di lavoro isolato con tutte le dipendenze: un ambiente di lavoro ci consente di permettere a chi poi replicherà il nostro lavoro di avere tutte le librerie informatiche di supporto installate e di quindi far funzionare il codice. Inoltre, ogni progetto ha il **suo** ambiente, cosi' i pacchetti di progetti diversi non si mescolano.

VA ESEGUITO UNA SOLA VOLTA ALL'INIZIO, POI MAI PIU'

Su Bash (terminale)

In [ ]:
conda create -n aind python=3.11
conda activate aind
# registra l'ambiente come kernel del notebook
python -m ipykernel install --user --name aind --display-name "Python (aind)"


In [ ]:
#In un file a parte "requirements.text" troverai le principali librerie usate in questo contesto e che va aggiornato quando ne aggiungi una. 

%%writefile requirements.txt
# --- pacchetti principali ---
numpy
pandas
matplotlib

# --- per scaricare i dataset da Kaggle ---
kagglehub

# --- opzionale: grafici dei valori mancanti (msno.matrix/bar/heatmap) ---
missingno


Ora, su VSCode lancia la cella:

In [ ]:
import sys
print('Python:', sys.version.split()[0])
print('Eseguibile:', sys.executable)   # deve puntare all'ambiente 'aind'

quando l'ambiente e' stabile, **congela le versioni esatte** con `pip freeze > requirements.txt`. Cosi' i risultati sono davvero riproducibili (stesse versioni = stesso comportamento).

In [ ]:
# %pip installa nell'ambiente del kernel attivo (piu' sicuro di !pip)
%pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('numpy      ', np.__version__)
print('pandas     ', pd.__version__)
print('matplotlib ', matplotlib.__version__)  #verifica

Best practise: riproducibilità dei risultati. Questa si ottiene usando un seed che lancia nelle stesse condizioni i run così da avere sempre lo stesso riusltato

In [ ]:
# seme casuale fisso: risultati ripetibili tra un'esecuzione e l'altra
np.random.seed(42)

## Come ottenere il dataset

Per la PRIMA volta, il modo piu' semplice e' scaricare a mano il CSV da
Kaggle (bottone "Download") e metterlo in una cartella data/ accanto a
questo file. Niente password, niente API: parti da qui.

dataset: https://www.kaggle.com/datasets/veeresh04/adnimerge

Per ora usiamo il file scaricato a mano:

In [ ]:
DATA_PATH = "data/adnimerge.csv"   # <-- TODO: metti qui il percorso reale

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Caricato:", df.shape)   # (righe, colonne)

In alternativa, puoi scaricarlo da codice (devi iscriverti a Kaggle (gratis) e serve un token (una password/chiave)): il token si crea in Account → Settings → sezione API → "Create New Token": scarica un kaggle.json con username e key. 
Quelli due sono le credenziali per l'autenticazione HTTP basic

In [ ]:
import kagglehub
path = kagglehub.dataset_download("veeresh04/adnimerge")
print(path)   # cartella dove e' finito il file

## 1. STUDIO BASE — NO MODIFICHE  (si guarda, non si tocca)

In [ ]:
print("Dimensioni:", df.shape)
df.info()          # tipi e non-nulli per colonna

In [ ]:
# TODO: guarda le prime righe e i tipi
df.head()
df.dtypes

# TODO: statistiche descrittive di TUTTE le colonne (numeriche + categoriche)
# suggerimento dal tuo metodo: df.describe(include="all")

In [ ]:
# ----------------------------------------------------------------------------
# 1b. VALORI MANCANTI
# ----------------------------------------------------------------------------
# Esempio: conteggio dei nulli per colonna, dal piu' problematico
nulli = df.isnull().sum().sort_values(ascending=False)
print(nulli.head(20))

# TODO: calcola la PERCENTUALE di nulli per colonna (nulli / numero righe * 100)
#       e stampa le 20 peggiori.

In [ ]:
# TODO (opzionale, se installi missingno): visualizza il pattern dei mancanti
# msno.matrix(df); plt.show()
# msno.bar(df);    plt.show()

In [ ]:
#--------------------------------------------------------------------------
# 1c. DUPLICATI E VALORI DELLE VARIABILI
# ----------------------------------------------------------------------------
# TODO: quante righe duplicate ci sono?  -> df.duplicated().sum()

In [ ]:
# TODO: guarda i valori di alcune colonne categoriche (quante volte compare ognuno).
#       Prova ad esempio con il sesso e la diagnosi:
# df["PTGENDER"].value_counts(dropna=False)
# df["DX"].value_counts(dropna=False)      # il nome esatto potrebbe variare: controllalo

In [ ]:
# TODO: quanti valori distinti ha una colonna?  -> df["PTGENDER"].nunique()

In [ ]:
# 1d. SOGGETTI E VISITE  (aggiunta rispetto al metodo base — importante per ADNI)
# In ADNI ogni soggetto (RID) puo' avere piu' visite. Vogliamo sapere QUANTI
# soggetti hanno 1, 2, 3, ... visite.
#
# Esempio gia' svolto:
visite_per_soggetto = df.groupby("RID").size()          # quante righe per RID
distribuzione = visite_per_soggetto.value_counts().sort_index()
distribuzione.index.name = "n_visite"
distribuzione.name = "n_soggetti"
print(distribuzione)

In [ ]:
# TODO: quanti soggetti UNICI ci sono in totale?  -> df["RID"].nunique()
# TODO: quanti hanno piu' di una visita?

2. PRIME MODIFICHE — CONSERVATIVO  (si lavora su una COPIA): regola d'oro del tuo metodo è non modificare l'originale. Lavora su una copia.

In [ ]:
df_clean = df.copy()

Esempio: rinominare una colonna verso un nome standard piu' chiaro.


In [ ]:
df_clean = df_clean.rename(columns={"PTGENDER": "GENDER"})

In [ ]:
# TODO: rimuovi le righe duplicate (se ce ne sono) -> df_clean.drop_duplicates(...)

In [ ]:
# TODO: scegli UNA colonna con molti nulli (dalla cella 1b) e NON cancellarla
#       subito: scrivi in un commento perche' potresti tenerla o toglierla.
#       (Il tuo metodo dice: la decisione dipende dal contesto, va motivata.)

In [ ]:
# TODO (facoltativo): standardizza una colonna di testo, es. togliere spazi /
#       uniformare maiuscole:  df_clean["col"] = df_clean["col"].str.strip()

3. RIEPILOGO DELLE DECISIONI  (la parte piu' importante)
   - Com'e' fatto il dataset? (righe, soggetti, visite)
   - Quali colonne sembrano chiave / utili e quali quasi vuote?
   - Quali problemi hai notato (formati strani, valori sospetti, duplicati)?
   - Cosa faresti nella prossima fase, e perche'?

Esempio di come annotare una decisione:
Colonna X: 78% mancante -> candidata alla rimozione, ma prima verifico
se i valori presenti sono concentrati in una sola coorte.